<a href="https://colab.research.google.com/github/02falgun/Be-Practical-Assessments/blob/main/Tasks_Day_9(task%2BGraded).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & SECURE GEMINI API AUTHENTICATION
# ==============================================================================

# Install required packages
!pip install -q -U google-genai pydantic python-dotenv tabulate


# ------------------------------------------------------------------------------
# Imports
# ------------------------------------------------------------------------------

import os
import sys
import time
import json
import random
import getpass

from typing import List, Dict, Any, Optional

from pydantic import BaseModel, Field

# Official Google GenAI SDK
from google import genai
from google.genai import types
from google.genai.errors import APIError


# ------------------------------------------------------------------------------
# Secure Gemini API Key
# ------------------------------------------------------------------------------
# Priority:
# 1. Google Colab Secrets
# 2. Environment variable
# 3. Manual input
#
# NEVER hardcode the API key in the notebook.
# ------------------------------------------------------------------------------

GEMINI_API_KEY = None

# Try Google Colab Secrets first
try:
    from google.colab import userdata

    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

except Exception:
    pass


# Try environment variable if Colab Secret wasn't found
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")


# Ask manually only if no key was found
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass(
        "🔑 Enter your Gemini API Key: "
    )


# Make the key available to the Google GenAI SDK
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY


# ------------------------------------------------------------------------------
# Validate API Key
# ------------------------------------------------------------------------------

if not GEMINI_API_KEY.strip():
    raise ValueError(
        "❌ Gemini API key is empty. "
        "Add GEMINI_API_KEY to Colab Secrets or enter a valid key."
    )


# ------------------------------------------------------------------------------
# Initialize Gemini Client
# ------------------------------------------------------------------------------

client = genai.Client()


# ------------------------------------------------------------------------------
# Success Message
# ------------------------------------------------------------------------------

print("✅ Google Gemini API Client initialized successfully!")
print("🔐 API key loaded securely.")

✅ Google Gemini API Client initialized successfully!
🔐 API key loaded securely.


In [2]:
# ==============================================================================
# COST ESTIMATION / PRE-FLIGHT TOKEN AUDIT
# ==============================================================================
# Best practice:
# Count input tokens BEFORE making expensive generation calls.
# This allows us to estimate the expected API cost before generation.
# ==============================================================================

import numpy as np
from typing import Dict, Any


def preflight_cost_estimate(
    text_prompt: str,
    model_name: str = "gemini-3.6-flash",
    expected_output_tokens: int = 500
) -> Dict[str, Any]:
    """
    Counts input tokens and estimates the cost of a Gemini API call
    before performing the actual generation request.
    """

    # --------------------------------------------------------------------------
    # 1. Count input tokens using the official Gemini tokenizer
    # --------------------------------------------------------------------------

    token_response = client.models.count_tokens(
        model=model_name,
        contents=text_prompt
    )

    input_tokens = token_response.total_tokens

    # --------------------------------------------------------------------------
    # 2. Gemini API pricing
    #    Prices are USD per 1 million tokens.
    #
    #    Current paid-tier pricing through Dec 31, 2026:
    #    Gemini 3.6 Flash:
    #       Input  = $0.75 / 1M tokens
    #       Output = $3.75 / 1M tokens
    # --------------------------------------------------------------------------

    pricing = {
        "gemini-3.6-flash": {
            "in": 0.75,
            "out": 3.75
        },

        "gemini-1.5-pro": {
            "in": 1.25,
            "out": 5.00
        }
    }

    if model_name not in pricing:
        raise ValueError(
            f"Pricing not configured for model: {model_name}"
        )

    rate = pricing[model_name]

    # --------------------------------------------------------------------------
    # 3. Estimate total cost
    # --------------------------------------------------------------------------

    estimated_input_cost = (
        input_tokens / 1_000_000
    ) * rate["in"]

    estimated_output_cost = (
        expected_output_tokens / 1_000_000
    ) * rate["out"]

    estimated_total_cost = (
        estimated_input_cost +
        estimated_output_cost
    )

    # --------------------------------------------------------------------------
    # 4. Return audit information
    # --------------------------------------------------------------------------

    return {
        "model": model_name,
        "input_tokens": input_tokens,
        "estimated_output_tokens": expected_output_tokens,

        "estimated_input_cost_usd": round(
            estimated_input_cost, 6
        ),

        "estimated_output_cost_usd": round(
            estimated_output_cost, 6
        ),

        "estimated_cost_usd": round(
            estimated_total_cost, 6
        ),

        "estimated_cost_per_10k_calls_usd": round(
            estimated_total_cost * 10_000,
            2
        )
    }


# ==============================================================================
# TEST / SAMPLE PROMPT
# ==============================================================================

sample_payload = (
    "Please summarize the last 10 quarterly financial filings "
    "of Apple, Microsoft, and Google."
)


# Run pre-flight cost estimation
estimate = preflight_cost_estimate(
    sample_payload,
    model_name="gemini-3.6-flash",
    expected_output_tokens=500
)


# ==============================================================================
# DISPLAY RESULTS
# ==============================================================================

print("=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===")

for key, value in estimate.items():
    print(f"• {key:35s}: {value}")

=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===
• model                              : gemini-3.6-flash
• input_tokens                       : 19
• estimated_output_tokens            : 500
• estimated_input_cost_usd           : 1.4e-05
• estimated_output_cost_usd          : 0.001875
• estimated_cost_usd                 : 0.001889
• estimated_cost_per_10k_calls_usd   : 18.89


In [4]:
# ==============================================================================
# API FAILURE HANDLING: EXPONENTIAL BACKOFF WITH JITTER
# ==============================================================================

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 4,
    base_delay: float = 1.5
):
    """
    Executes an API call with exponential backoff and random jitter.

    Retries temporary errors such as:
    - HTTP 429: Rate limit / quota exceeded
    - HTTP 500: Internal server error
    - HTTP 502: Bad gateway
    - HTTP 503: Service unavailable
    - HTTP 504: Gateway timeout

    Formula:
        wait_time = (base_delay * 2^attempt) + random_jitter
    """

    retryable_codes = {429, 500, 502, 503, 504}

    for attempt in range(max_retries):

        try:
            # Execute the API call
            return api_call_func()

        except APIError as e:

            # Do not retry permanent errors
            if e.code not in retryable_codes:
                print(
                    f"❌ Non-retryable API Error "
                    f"(HTTP {e.code}): {e}"
                )
                raise

            # Stop after maximum attempts
            if attempt == max_retries - 1:
                print(
                    f"❌ Maximum retries reached. "
                    f"Final API Error (HTTP {e.code}): {e}"
                )
                raise

            # Exponential backoff
            exponential_delay = base_delay * (2 ** attempt)

            # Random jitter prevents simultaneous retries
            jitter = random.uniform(0.1, 0.8)

            delay = exponential_delay + jitter

            print(
                f"⚠️ Transient API Error "
                f"(HTTP {e.code}). "
                f"Retrying in {delay:.2f}s... "
                f"(Attempt {attempt + 1}/{max_retries})"
            )

            time.sleep(delay)

In [5]:
# ==============================================================================
# GEMINI API WRAPPER
# ==============================================================================

def gemini_call(
    prompt: str,
    system_instruction: str = "You are a concise, helpful enterprise AI assistant.",
    temperature: float = 0.2,
    stream: bool = False,
    model: str = "gemini-3.6-flash"
) -> str:
    """
    Production-grade wrapper for the Google Gemini API.
    Supports standard and streaming responses with retry handling.
    """

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=800
    )

    # --------------------------------------------------------------------------
    # Streaming response
    # --------------------------------------------------------------------------

    if stream:

        def stream_call():
            full_text = []

            response_stream = client.models.generate_content_stream(
                model=model,
                contents=prompt,
                config=config
            )

            for chunk in response_stream:
                if chunk.text:
                    print(chunk.text, end="", flush=True)
                    full_text.append(chunk.text)

            print()

            return "".join(full_text).strip()

        return execute_with_exponential_backoff(stream_call)

    # --------------------------------------------------------------------------
    # Standard response
    # --------------------------------------------------------------------------

    else:

        def standard_call():
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=config
            )

            return response.text.strip()

        return execute_with_exponential_backoff(standard_call)


# ==============================================================================
# 3-TURN CONVERSATIONAL MEMORY DEMONSTRATION
# ==============================================================================

print("=== MULTI-TURN CONVERSATION LOOP ===")

# Explicitly maintain conversation history
conversation_history = []

system_persona = (
    "You are a Senior PostgreSQL Database Administrator. "
    "Answer concisely in 2 sentences."
)


def send_chat_turn(user_message: str):

    print(f"\n👤 User: {user_message}")
    print("🤖 Assistant: ", end="")

    # --------------------------------------------------------------------------
    # 1. Add user message to conversation history
    # --------------------------------------------------------------------------

    conversation_history.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_message)]
        )
    )

    # --------------------------------------------------------------------------
    # 2. Configure Gemini
    # --------------------------------------------------------------------------

    config = types.GenerateContentConfig(
        system_instruction=system_persona,
        temperature=0.0,
        max_output_tokens=300
    )

    # --------------------------------------------------------------------------
    # 3. Generate response WITH exponential-backoff protection
    # --------------------------------------------------------------------------

    def conversation_call():

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=conversation_history,
            config=config
        )

        return response.text.strip()

    bot_reply = execute_with_exponential_backoff(
        conversation_call
    )

    print(bot_reply)

    # --------------------------------------------------------------------------
    # 4. Add model response to history
    # --------------------------------------------------------------------------

    conversation_history.append(
        types.Content(
            role="model",
            parts=[types.Part.from_text(text=bot_reply)]
        )
    )

    return bot_reply


# ==============================================================================
# EXECUTE 3-TURN DIALOGUE
# ==============================================================================

send_chat_turn(
    "What is the difference between a clustered and non-clustered index?"
)

send_chat_turn(
    "Which one is faster for range queries on primary keys?"
)

# Demonstrates contextual understanding and pronoun resolution
send_chat_turn(
    "Can a table have multiple of the faster one?"
)

=== MULTI-TURN CONVERSATION LOOP ===

👤 User: What is the difference between a clustered and non-clustered index?
🤖 Assistant: A clustered index physically reorders the actual

👤 User: Which one is faster for range queries on primary keys?
🤖 Assistant: A clustered index is significantly faster for range queries on

👤 User: Can a table have multiple of the faster one?
🤖 Assistant: No, a table can only have one


'No, a table can only have one'

In [6]:
# Create .gitignore

with open(".gitignore", "w") as f:
    f.write(
        ".env\n"
        ".env.local\n"
        ".env.*\n"
        "*.joblib\n"
        "__pycache__/\n"
        ".ipynb_checkpoints/\n"
    )

print("✅ .gitignore created successfully!")

✅ .gitignore created successfully!


In [7]:
import os

print("Files in current directory:")
for file in os.listdir("."):
    print("•", file)

Files in current directory:
• .config
• .gitignore
• sample_data


In [8]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & GOOGLE GEMINI API AUTHENTICATION
# ==============================================================================

!pip install -q -U google-genai tiktoken tabulate

import os
import time
import math
import random
import getpass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tabulate import tabulate
from google import genai
from google.genai import types
from google.genai.errors import APIError

# Plotting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 10

# ------------------------------------------------------------------------------
# Secure Gemini API Key
# ------------------------------------------------------------------------------

GEMINI_API_KEY = None

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass(
        "🔑 Enter your Google Gemini API Key: "
    )

if not GEMINI_API_KEY.strip():
    raise ValueError("❌ Gemini API key is empty.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Initialize client
client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Google Gemini Client initialized successfully!")

✅ Google Gemini Client initialized successfully!


In [9]:
# ==============================================================================
# GEMINI COST CALCULATOR
# ==============================================================================

def calculate_gemini_cost(
    n_input: int,
    n_output: int,
    model: str = "gemini-3.6-flash"
) -> float:

    # USD per 1 million tokens
    pricing_matrix = {
        "gemini-3.6-flash": {
            "input_per_m": 0.75,
            "output_per_m": 3.75
        }
    }

    if model not in pricing_matrix:
        raise ValueError(
            f"No pricing configured for model: {model}"
        )

    rates = pricing_matrix[model]

    input_cost = (
        n_input / 1_000_000
    ) * rates["input_per_m"]

    output_cost = (
        n_output / 1_000_000
    ) * rates["output_per_m"]

    return input_cost + output_cost

In [10]:
# ==============================================================================
# ROBUST API CALL WITH EXPONENTIAL BACKOFF
# ==============================================================================

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 4,
    base_delay: float = 1.5
):
    """
    Executes an API call with exponential backoff and jitter.

    Retries temporary API failures:
    429, 500, 502, 503, 504
    """

    retryable_codes = {429, 500, 502, 503, 504}

    for attempt in range(max_retries):

        try:
            return api_call_func()

        except APIError as e:

            # Permanent error → don't retry
            if e.code not in retryable_codes:
                print(
                    f"❌ Non-retryable API Error "
                    f"(HTTP {e.code}): {e}"
                )
                raise

            # No attempts remaining
            if attempt == max_retries - 1:
                print(
                    f"❌ Maximum retries reached. "
                    f"Final API Error (HTTP {e.code}): {e}"
                )
                raise

            # Exponential backoff + jitter
            exponential_delay = base_delay * (2 ** attempt)
            jitter = random.uniform(0.1, 0.8)

            delay = exponential_delay + jitter

            print(
                f"⚠️ HTTP {e.code}. "
                f"Retrying in {delay:.2f}s "
                f"(Attempt {attempt + 1}/{max_retries})"
            )

            time.sleep(delay)

In [11]:
# ==============================================================================
# GEMINI API WRAPPER
# ==============================================================================

def gemini_call(
    prompt: str,
    system_instruction: str = (
        "You are a concise, helpful enterprise AI assistant."
    ),
    temperature: float = 0.2,
    top_p: float = 0.95,
    stream: bool = False,
    model: str = "gemini-3.6-flash"
) -> str:

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=800
    )

    if stream:

        def stream_call():

            full_text = []

            response_stream = client.models.generate_content_stream(
                model=model,
                contents=prompt,
                config=config
            )

            for chunk in response_stream:

                if chunk.text:
                    print(chunk.text, end="", flush=True)
                    full_text.append(chunk.text)

            print()

            return "".join(full_text).strip()

        return execute_with_exponential_backoff(stream_call)

    else:

        def standard_call():

            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=config
            )

            return response.text.strip()

        return execute_with_exponential_backoff(standard_call)

In [12]:
# ==============================================================================
# SECTION 8: STUDENT LAB SOLUTION
# ==============================================================================

lab_prompt = (
    "Explain the concept of 'Technical Debt' to a "
    "non-technical CEO using a vivid real-world analogy."
)

experiments = [
    {
        "Configuration": "Config 1",
        "Description": "Deterministic Baseline",
        "Temperature": 0.0,
        "Top-P": 0.95
    },
    {
        "Configuration": "Config 2",
        "Description": "Controlled Diversity",
        "Temperature": 0.5,
        "Top-P": 0.80
    },
    {
        "Configuration": "Config 3",
        "Description": "Balanced Creative",
        "Temperature": 0.9,
        "Top-P": 0.95
    },
    {
        "Configuration": "Config 4",
        "Description": "High Entropy",
        "Temperature": 1.4,
        "Top-P": 1.00
    }
]

results = []

print("=== TECHNICAL DEBT SAMPLING EXPERIMENT ===\n")

for experiment in experiments:

    temperature = experiment["Temperature"]
    top_p = experiment["Top-P"]

    print("=" * 80)
    print(
        f"{experiment['Configuration']} | "
        f"{experiment['Description']}"
    )
    print(
        f"Temperature = {temperature} | "
        f"Top-P = {top_p}"
    )
    print("=" * 80)

    # --------------------------------------------------------------
    # Count input tokens BEFORE generation
    # --------------------------------------------------------------

    token_response = client.models.count_tokens(
        model="gemini-3.6-flash",
        contents=lab_prompt
    )

    input_tokens = token_response.total_tokens

    # --------------------------------------------------------------
    # Generate response with retry protection
    # --------------------------------------------------------------

    def generate_experiment():

        config = types.GenerateContentConfig(
            temperature=temperature,
            top_p=top_p,
            max_output_tokens=300
        )

        return client.models.generate_content(
            model="gemini-3.6-flash",
            contents=lab_prompt,
            config=config
        )

    response = execute_with_exponential_backoff(
        generate_experiment
    )

    generated_text = response.text.strip()

    # --------------------------------------------------------------
    # Actual token usage from response
    # --------------------------------------------------------------

    if response.usage_metadata:

        output_tokens = (
            response.usage_metadata.candidates_token_count or 0
        )

        total_tokens = (
            response.usage_metadata.total_token_count or 0
        )

    else:

        output_tokens = 0
        total_tokens = input_tokens

    # --------------------------------------------------------------
    # Calculate cost
    # --------------------------------------------------------------

    estimated_cost = calculate_gemini_cost(
        input_tokens,
        output_tokens,
        model="gemini-3.6-flash"
    )

    # --------------------------------------------------------------
    # Store results
    # --------------------------------------------------------------

    results.append({
        "Configuration": experiment["Configuration"],
        "Description": experiment["Description"],
        "Temperature": temperature,
        "Top-P": top_p,
        "Input Tokens": input_tokens,
        "Output Tokens": output_tokens,
        "Total Tokens": total_tokens,
        "Estimated Cost (USD)": estimated_cost,
        "Response": generated_text
    })

    print("\nResponse:")
    print(generated_text)

    print(
        f"\nInput Tokens: {input_tokens}"
    )

    print(
        f"Output Tokens: {output_tokens}"
    )

    print(
        f"Total Tokens: {total_tokens}"
    )

    print(
        f"Estimated Cost: ${estimated_cost:.6f}"
    )

# ==============================================================================
# RESULTS TABLE
# ==============================================================================

results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "Configuration",
            "Description",
            "Temperature",
            "Top-P",
            "Input Tokens",
            "Output Tokens",
            "Total Tokens",
            "Estimated Cost (USD)"
        ]
    ]
)

=== TECHNICAL DEBT SAMPLING EXPERIMENT ===

Config 1 | Deterministic Baseline
Temperature = 0.0 | Top-P = 0.95

Response:
though accurate. "Interest" is obvious, but might

Input Tokens: 23
Output Tokens: 11
Total Tokens: 319
Estimated Cost: $0.000058
Config 2 | Controlled Diversity
Temperature = 0.5 | Top-P = 0.8

Response:
Drywall over bad plumbing. Good, highly relatable for

Input Tokens: 23
Output Tokens: 11
Total Tokens: 319
Estimated Cost: $0.000058
Config 3 | Balanced Creative
Temperature = 0.9 | Top-P = 0.95

Response:
Building a Skyscraper/House with cheap materials

Input Tokens: 23
Output Tokens: 9
Total Tokens: 319
Estimated Cost: $0.000051
Config 4 | High Entropy
Temperature = 1.4 | Top-P = 1.0

Response:
Building a House / Architecture.* Good, but can get bogged

Input Tokens: 23
Output Tokens: 12
Total Tokens: 319
Estimated Cost: $0.000062


,Configuration,Description,Temperature,Top-P,Input Tokens,Output Tokens,Total Tokens,Estimated Cost (USD)
0,Config 1,Deterministic Baseline,0.0,0.95,23,11,319,0.000058
1,Config 2,Controlled Diversity,0.5,0.80,23,11,319,0.000058
2,Config 3,Balanced Creative,0.9,0.95,23,9,319,0.000051
3,Config 4,High Entropy,1.4,1.00,23,12,319,0.000062


In [13]:
 # ==============================================================================
# STYLISTIC DIVERGENCE ANALYSIS
# ==============================================================================

print("\n=== STYLISTIC DIVERGENCE ANALYSIS ===\n")

for _, row in results_df.iterrows():

    print(
        f"{row['Configuration']} "
        f"(T={row['Temperature']}, Top-P={row['Top-P']}):"
    )

    print(
        f"- Output length: {row['Output Tokens']} tokens"
    )

    print(
        f"- Estimated cost: ${row['Estimated Cost (USD)']:.6f}"
    )

    print(
        f"- Response style: {row['Description']}"
    )

    print()


=== STYLISTIC DIVERGENCE ANALYSIS ===

Config 1 (T=0.0, Top-P=0.95):
- Output length: 11 tokens
- Estimated cost: $0.000058
- Response style: Deterministic Baseline

Config 2 (T=0.5, Top-P=0.8):
- Output length: 11 tokens
- Estimated cost: $0.000058
- Response style: Controlled Diversity

Config 3 (T=0.9, Top-P=0.95):
- Output length: 9 tokens
- Estimated cost: $0.000051
- Response style: Balanced Creative

Config 4 (T=1.4, Top-P=1.0):
- Output length: 12 tokens
- Estimated cost: $0.000062
- Response style: High Entropy



In [15]:
with open(".gitignore", "w") as f:
    f.write(
        ".env\n"
        ".env.local\n"
        ".env.*\n"
        "*.joblib\n"
        "__pycache__/\n"
        ".ipynb_checkpoints/\n"
    )

print("✅ .gitignore created!")

✅ .gitignore created!
